# JeleBondhu · Corrected Colab Calibration and Validation (v3)

**Run in order.** First validate station/PAI alignment and fisheries events, then optionally build monthly climatology. This notebook does **not** deploy GitHub Pages, issue fishing safety clearance, or produce validated seven-day species-specific PFZ forecasts.

**Main correction:** never mix `stations-2025.json` (list) with an independently ordered seasonal PAI file, and never write zero for missing PAI. The validation script now supports a station-ID dictionary or an aligned positional list.

**Security:** Rotate any previously shared Copernicus password. Credentials are entered only into a hidden prompt, never committed.

## A. Setup — mount Drive, clone/update GitHub, install packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, sys, subprocess, json, numpy as np
REPO = Path("/content/jelebondhu")
URL = "https://github.com/amithasanbd/JeleBondhu_WebApp_NRT_v1.0.git"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
else:
    print("Existing repository found; run git pull only if there are no local edits.")
    status = subprocess.run(["git","status","--porcelain"],cwd=REPO,capture_output=True,text=True,check=True)
    if not status.stdout.strip():
        subprocess.run(["git","pull","--ff-only"],cwd=REPO,check=True)
    else:
        print("Local edits detected; skipping pull to avoid overwriting work.")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "backend"))
subprocess.run([sys.executable,"-m","pip","-q","install","numpy","pandas","openpyxl","scipy","xarray","netCDF4","h5netcdf","copernicusmarine","pyarrow"],check=True)
print("Repository:", REPO)

## B. Apply the repaired validation script (only if your GitHub repository has not yet been updated)

In [ ]:
# Upload 'validate_events.py' supplied with this repair to Colab's /content folder,
# OR replace the script in GitHub before running the setup cell.
from pathlib import Path
import shutil
uploaded = Path("/content/validate_events.py")
target = REPO / "scripts/validate_events.py"
if uploaded.exists():
    shutil.copy2(uploaded, target)
    print("Installed corrected validation script:", target)
else:
    from importlib.machinery import SourceFileLoader
    mod = SourceFileLoader("validation_check", str(target)).load_module()
    if not hasattr(mod, "seasonal_pai_values"):
        raise RuntimeError("Old validator detected. Upload corrected validate_events.py into /content and rerun this cell.")
    print("Corrected validation script already present.")

## C. Check station alignment and PAI units (required before validation)

In [ ]:
from pathlib import Path
import json, numpy as np
station_path = REPO / "data/static/stations.json"
pai_path = REPO / "data/static/thesis_pai_2025.json"
stations = json.loads(station_path.read_text())
thesis = json.loads(pai_path.read_text())
assert isinstance(stations,dict) and all(k in stations for k in ("lat","lon","depth_m")), "Unexpected stations.json schema"
n = len(stations["lat"])
assert len(stations["lon"]) == len(stations["depth_m"]) == n
assert n > 0
assert "pai" in thesis and isinstance(thesis["pai"],dict)
print("Station count:",n)
print("PAI unit:", thesis.get("unit","MISSING"))
print("Seasons:",list(thesis["pai"]))
for season, data in thesis["pai"].items():
    if isinstance(data,list):
        assert len(data)==n, f"{season} has {len(data)} PAI values but {n} stations"
        print(season, "aligned positional list", len(data))
    elif isinstance(data,dict):
        print(season, "ID-keyed dictionary", len(data), "keys")
    else:
        raise ValueError(f"Invalid PAI structure: {season}")
print("No files were overwritten.")

## D. Optional: import your `stations-2025.json` list SAFELY

Only run if you need to replace the station layer. This section **refuses to overwrite** the existing station or PAI file unless you deliberately change `ENABLE_REPLACEMENT=True`. It preserves missing values as missing, requires unique station IDs, and keeps a single consistent row order.

**Warning:** The source list's `seasons` must actually contain station-specific PAI values. If it does not, retain the existing thesis PAI file.

In [ ]:
from pathlib import Path
import json, math, shutil
SOURCE = Path("/content/drive/MyDrive/MSc_Thesis/0.PFZ Thesis 2026/0.JeleBondhu Outputs/JeleBondhu_Extraction/final/stations-2025.json")
ENABLE_REPLACEMENT = False  # Keep False unless you verified source IDs and PAI.
if not SOURCE.exists():
    print("Optional source not found; existing repository station and PAI files retained.")
else:
    raw = json.loads(SOURCE.read_text())
    if not isinstance(raw,list):
        print("Source is not a list; inspect its schema manually. No changes made.")
    else:
        def number(value):
            try:
                v=float(value)
                return v if math.isfinite(v) else None
            except (TypeError,ValueError):
                return None
        ids = [str(x.get("id",x.get("station_id",""))) for x in raw]
        if not all(ids) or len(set(ids))!=len(ids):
            raise ValueError("Missing or duplicate station IDs; refusing to convert")
        if any(number(x.get("lat")) is None or number(x.get("lon")) is None for x in raw):
            raise ValueError("Missing station coordinates; refusing to replace")
        new_st = {
            "n":len(raw), "id":ids,
            "lat":[number(x.get("lat")) for x in raw],
            "lon":[number(x.get("lon")) for x in raw],
            "depth_m":[number(x.get("depth_m",x.get("depth"))) for x in raw],
        }
        seasons=("Pre-monsoon","Monsoon","Post-monsoon","Winter")
        new_pai={"unit":"PAI fraction (0-1)","key_type":"station_id","pai":{}}
        for season in seasons:
            vals={}
            for sid,item in zip(ids,raw):
                v=number((item.get("seasons") or {}).get(season,{}).get("pai")
                         if isinstance((item.get("seasons") or {}).get(season,{}),dict) else None)
                if v is not None and 0<=v<=1: vals[sid]=v
            new_pai["pai"][season]=vals
        coverage={s:len(v) for s,v in new_pai["pai"].items()}
        print("Candidate station count:",len(ids),"PAI coverage:",coverage)
        print("Current repository stations:",len(stations["lat"]))
        if ENABLE_REPLACEMENT:
            if len(raw)!=len(stations["lat"]) or any(x!=len(raw) for x in coverage.values()):
                raise ValueError("Coverage or station-count mismatch; refusing replacement")
            for path in (station_path,pai_path):
                shutil.copy2(path, path.with_suffix(path.suffix+".bak"))
            station_path.write_text(json.dumps(new_st,ensure_ascii=False))
            pai_path.write_text(json.dumps(new_pai,ensure_ascii=False))
            print("Replaced after validation; backups created.")
        else:
            print("DRY RUN ONLY. Existing reference files are unchanged.")

## E. Event-grouped fisheries validation — run this first

In [ ]:
from pathlib import Path
import subprocess,sys,json
XLSX = Path("/content/drive/MyDrive/MSc_Thesis/0.PFZ Thesis 2026/Final Datasets, JeleBondhu/JeleBondhu_Commercial_Fisheries_PlotReady_EEZ_20261124.xlsx")
if not XLSX.exists():
    raise FileNotFoundError(f"Upload your PlotReady workbook to this exact Drive location or edit XLSX:\n{XLSX}")
cmd=[sys.executable,str(REPO/"scripts/validate_events.py"),"--xlsx",str(XLSX)]
subprocess.run(cmd,cwd=REPO,check=True)
out=REPO/"data/static/validation_summary.json"
print("Validation output:",out,"bytes:",out.stat().st_size)

## F. Optional: Copernicus monthly climatology (not needed to fix KeyError)

In [ ]:
import getpass, copernicusmarine as cm
username=input("Copernicus username: ").strip()
password=getpass.getpass("NEW Copernicus password: ")
cm.login(username=username,password=password,force_overwrite=True)
del password
print("Login request completed. Confirm product and variable availability before downloading.")

In [ ]:
# IMPORTANT: This is a SMALL one-year test, not the final 1995-2025 reference.
# Confirm that both thetao and zos are present in the dataset you choose.
import xarray as xr, pandas as pd, numpy as np
DATASET="cmems_mod_glo_phy_my_0.083deg_P1M-m"
ds=cm.open_dataset(dataset_id=DATASET,variables=["thetao","zos"],
    minimum_longitude=88.0,maximum_longitude=93.2,
    minimum_latitude=17.2,maximum_latitude=23.3,
    start_datetime="2020-01-01",end_datetime="2020-12-31")
print(ds)

In [ ]:
# Sample the EXACT stations.json coordinates, not config/ports.json.
# Run only after the previous Copernicus cell succeeds.
from jelebondhu.reference import build_reference
import xarray as xr, pandas as pd, numpy as np
la=xr.DataArray(np.asarray(stations["lat"],float),dims="station")
lo=xr.DataArray(np.asarray(stations["lon"],float),dims="station")
rows=[]
for name,var in (("sst","thetao"),("zos","zos")):
    da=ds[var]
    if "depth" in da.dims: da=da.isel(depth=0)
    clim=da.groupby("time.month").mean("time")
    pts=clim.sel(latitude=la,longitude=lo,method="nearest")
    vals=np.asarray(pts.values)
    if vals.shape!=(12,n):
        raise ValueError(f"Unexpected dimensions for {name}: {vals.shape}")
    rows.append(pd.DataFrame(vals.T,columns=range(1,13)).assign(station_idx=range(n)).melt(
        "station_idx",var_name="month",value_name=name))
df=rows[0].merge(rows[1],on=["station_idx","month"])
print("Missing fractions:",df[["sst","zos"]].isna().mean().to_dict())
print("ONE-YEAR TEST ONLY. Do not publish as 1995-2025 climatology.")
# For the final reference, repeat the download for the full period in yearly chunks,
# verify complete temporal coverage, and only then call build_reference.

## G. Save verified outputs to Drive; then upload to GitHub

In [ ]:
from pathlib import Path
import shutil
destination=Path("/content/drive/MyDrive/JeleBondhu/Verified_Outputs")
destination.mkdir(parents=True,exist_ok=True)
for rel in ("data/static/validation_summary.json",):
    src=REPO/rel
    if src.exists():
        dest=destination/src.name
        shutil.copy2(src,dest)
        print("Saved:",dest)
print("Upload verified validation_summary.json to data/static/ in GitHub.")
print("Do NOT set Filter 1 to frozen based only on a climatology file.")